# Clasificación de dígitos escritos a mano con MNIST

Este cuaderno prepara los datos de MNIST, entrena una red neuronal, mide su desempeño, permite probar una imagen propia y termina mostrando la arquitectura en una tabla.

**Recorrido:** carga y normalización → definición y entrenamiento → evaluación y predicción → tabla de capas.


## Ejecutar localmente en Visual Studio Code

1. Instala **Python 3.12 de 64 bits**. El entorno actual usa Python 3.14, que no es compatible con TensorFlow 2.21; Python 3.12 es una versión compatible y recomendada.
2. En VS Code, instala las extensiones **Python** y **Jupyter** de Microsoft.
3. Abre la carpeta del proyecto en VS Code y crea el entorno virtual desde la terminal integrada.

**Windows PowerShell:**
```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

**Linux o macOS:**
```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

4. Abre `CodeTAM.ipynb`, selecciona el kernel de Python dentro de `.venv` y ejecuta las celdas en orden. La primera carga de MNIST necesita conexión a Internet; el conjunto queda en caché para las ejecuciones siguientes.

Las dependencias están declaradas en `requirements.txt`. En VS Code se necesita la extensión Jupyter para mostrar la interfaz de carga con `ipywidgets`.


In [ ]:
# BLOQUE 1. Detectar temprano una versión de Python no compatible con TensorFlow.
import sys

if not ((3, 10) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(
        "Este cuaderno requiere Python 3.10–3.13 por TensorFlow 2.21; Python 3.14 no es compatible. "
        "Usa Python 3.12 y selecciona ese entorno como kernel en VS Code."
    )

# BLOQUE 2. Importar las bibliotecas usadas en el cuaderno.
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

# BLOQUE 3. Fijar semillas para facilitar la reproducción del experimento.
np.random.seed(42)
tf.random.set_seed(42)


## 2. Carga y preparación de MNIST

MNIST contiene imágenes de entrenamiento y prueba de dígitos del 0 al 9. Cada imagen original mide 28 × 28 píxeles. La red recibe un vector de 784 valores, y cada píxel se escala de `[0, 255]` a `[0, 1]`. Las etiquetas se conservan como enteros.


In [ ]:
# BLOQUE 1. Descargar (o cargar desde la caché local) las imágenes y etiquetas.
(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.mnist.load_data()

# BLOQUE 2. Aplanar cada imagen y normalizar sus píxeles al intervalo [0, 1].
x_train = x_train_raw.reshape((-1, 28 * 28)).astype(np.float32) / 255.0
x_test = x_test_raw.reshape((-1, 28 * 28)).astype(np.float32) / 255.0
del x_train_raw, x_test_raw

# BLOQUE 3. Comprobar formas, rango de píxeles y etiquetas enteras del conjunto.
assert x_train.shape == (60_000, 784)
assert x_test.shape == (10_000, 784)
assert np.issubdtype(y_train.dtype, np.integer)
assert int(y_train.min()) == 0 and int(y_train.max()) == 9
assert float(x_train.min()) >= 0.0 and float(x_train.max()) <= 1.0

print(f"Imágenes de entrenamiento: {x_train.shape[0]:,}")
print(f"Imágenes de prueba: {x_test.shape[0]:,}")
print(f"Forma de cada entrada: {x_train.shape[1]} características")
print(f"Forma de las etiquetas: {y_train.shape}; valores posibles: 0 a 9")

# BLOQUE 4. Mostrar una imagen para verificar la carga y su etiqueta.
plt.figure(figsize=(3, 3))
plt.imshow(x_train[0].reshape((28, 28)), cmap="gray", vmin=0, vmax=1)
plt.title(f"Etiqueta real: {int(y_train[0])}")
plt.axis("off")
plt.show()


## 3. Definición del modelo

La entrada tiene 784 valores. La red usa dos capas ocultas (`64` y `16` neuronas, ambas con `ReLU`) y una capa de salida de `10` unidades, una por dígito, con `softmax` para producir probabilidades.


In [ ]:
# BLOQUE 1. Declarar la entrada: una imagen aplanada en 784 píxeles.
inputs = tf.keras.Input(shape=(784,), name="pixels")

# BLOQUE 2. Construir las capas ocultas y especificar su activación.
hidden_1 = tf.keras.layers.Dense(64, activation="relu", name="hidden_1")(inputs)
hidden_2 = tf.keras.layers.Dense(16, activation="relu", name="hidden_2")(hidden_1)

# BLOQUE 3. Producir una probabilidad para cada una de las diez clases.
outputs = tf.keras.layers.Dense(
    10, activation="softmax", name="class_probabilities"
)(hidden_2)
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="mnist_digit_classifier")

# BLOQUE 4. Mostrar el resumen estándar de Keras.
model.summary()


## 4. Elegir la función de pérdida correctamente

`mnist.load_data()` entrega cada etiqueta como un entero entre `0` y `9`.

| Función | Formato de etiqueta | ¿Hay que convertir a one-hot? |
|---|---|---|
| `sparse_categorical_crossentropy` | Entero, por ejemplo `7` | No |
| `categorical_crossentropy` | Vector one-hot, por ejemplo `[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]` | Sí, antes de entrenar |

**Aclaración importante:** `sparse_categorical_crossentropy` no convierte el número en un vector; recibe directamente la etiqueta entera. `categorical_crossentropy` necesita que la etiqueta ya esté representada como vector one-hot. En este cuaderno usamos la variante *sparse* y mantenemos `y_train` y `y_test` como enteros.

La salida del modelo usa `softmax`, por eso la pérdida se configura con `from_logits=False`.


## 5. Compilación y entrenamiento

Se usa Adam, la pérdida sparse adecuada para las etiquetas enteras y exactitud dispersa como métrica. Una parte de los datos de entrenamiento se reserva para validación.


In [ ]:
# BLOQUE 1. Configurar optimizador, pérdida y métrica para etiquetas enteras.
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

# BLOQUE 2. Entrenar y guardar el historial de métricas por época.
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)


## 6. Evaluación y predicción de ejemplo

La evaluación usa el conjunto de prueba, que no participó en el entrenamiento. Después se muestra una imagen de prueba junto con la clase predicha y la probabilidad más alta.


In [ ]:
# BLOQUE 1. Medir la pérdida y exactitud sobre imágenes no vistas durante el entrenamiento.
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Pérdida en prueba: {test_loss:.4f}")
print(f"Exactitud en prueba: {test_accuracy:.2%}")

# BLOQUE 2. Predecir una imagen de prueba y comprobar el formato de salida.
sample_index = 0
probabilities = model.predict(x_test[sample_index : sample_index + 1], verbose=0)[0]
predicted_digit = int(np.argmax(probabilities))
actual_digit = int(y_test[sample_index])
assert probabilities.shape == (10,)
assert np.isclose(float(probabilities.sum()), 1.0, atol=1e-5)

print(f"Etiqueta real: {actual_digit}")
print(f"Predicción: {predicted_digit}")
print(f"Probabilidad de la predicción: {float(probabilities[predicted_digit]):.2%}")

# BLOQUE 3. Visualizar la imagen usada en la predicción.
plt.figure(figsize=(3, 3))
plt.imshow(x_test[sample_index].reshape((28, 28)), cmap="gray", vmin=0, vmax=1)
plt.title(f"Real: {actual_digit} | Predicción: {predicted_digit}")
plt.axis("off")
plt.show()


## 7. Probar una imagen propia

En Google Colab se abre su selector de archivos. En VS Code, la celda muestra un selector `ipywidgets.FileUpload`; el entorno virtual debe tener instalados los paquetes de `requirements.txt` y la extensión Jupyter debe estar activa. Se puede subir una o varias imágenes PNG o JPG. El preprocesamiento convierte cada imagen a escala de grises, invierte el fondo si hace falta, recorta el dígito, lo centra en un lienzo de 28 × 28 y lo normaliza como MNIST.


In [ ]:
# BLOQUE 1. Importar las herramientas de imagen y detectar si el cuaderno corre en Colab.
from io import BytesIO
from PIL import Image, ImageOps

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None


# BLOQUE 2. Aplicar a la imagen cargada el formato esperado por el modelo.
def preprocess_uploaded_digit(image):
    # Convertir a escala de grises y aumentar el contraste.
    grayscale = ImageOps.autocontrast(ImageOps.grayscale(image))
    pixels = np.asarray(grayscale, dtype=np.uint8)

    # MNIST usa un dígito claro sobre fondo oscuro; invertir si el fondo es claro.
    if float(pixels.mean()) > 127.0:
        pixels = 255 - pixels

    # Recortar el fondo y rechazar archivos que no contienen un trazo visible.
    mask = pixels > 20
    if not bool(mask.any()):
        raise ValueError("No se encontró un dígito visible en la imagen.")
    rows, columns = np.where(mask)
    cropped = Image.fromarray(
        pixels[rows.min() : rows.max() + 1, columns.min() : columns.max() + 1]
    )

    # Mantener la proporción, ajustar a 20 × 20 como máximo y centrar en 28 × 28.
    scale = 20.0 / max(cropped.size)
    resized_size = tuple(max(1, int(round(size * scale))) for size in cropped.size)
    resampling = getattr(Image, "Resampling", Image).LANCZOS
    cropped = cropped.resize(resized_size, resampling)
    canvas = Image.new("L", (28, 28), color=0)
    canvas.paste(cropped, ((28 - cropped.width) // 2, (28 - cropped.height) // 2))

    normalized = np.asarray(canvas, dtype=np.float32) / 255.0
    return normalized.reshape((1, 784)), normalized


# BLOQUE 3. Predecir y mostrar las imágenes recibidas por cualquiera de los selectores.
def predict_uploaded_images(uploaded_items):
    for filename, image_bytes in uploaded_items:
        try:
            with Image.open(BytesIO(bytes(image_bytes))) as uploaded_image:
                input_vector, preview = preprocess_uploaded_digit(uploaded_image)
        except (OSError, ValueError) as error:
            print(f"No se pudo procesar {filename}: {error}")
            continue

        image_probabilities = model.predict(input_vector, verbose=0)[0]
        predicted_digit = int(np.argmax(image_probabilities))
        confidence = float(image_probabilities[predicted_digit])

        plt.figure(figsize=(3, 3))
        plt.imshow(preview, cmap="gray", vmin=0, vmax=1)
        plt.title(f"{filename}: predicción {predicted_digit} ({confidence:.1%})")
        plt.axis("off")
        plt.show()


# BLOQUE 4. Usar el selector nativo de Colab o el widget compatible con VS Code/Jupyter.
if colab_files is not None:
    uploaded_images = colab_files.upload()
    predict_uploaded_images(uploaded_images.items())
else:
    try:
        import ipywidgets as widgets
    except ImportError as error:
        raise ImportError(
            "Instala las dependencias con `python -m pip install -r requirements.txt`."
        ) from error

    upload_widget = widgets.FileUpload(accept="image/*", multiple=True)
    upload_output = widgets.Output()

    def on_upload(change):
        selected_files = change["new"]
        if isinstance(selected_files, dict):
            # Formato de ipywidgets 7: nombre -> información del archivo.
            uploaded_items = [
                (filename, item["content"])
                for filename, item in selected_files.items()
            ]
        else:
            # Formato de ipywidgets 8: secuencia de diccionarios.
            uploaded_items = [
                (item["name"], item["content"])
                for item in selected_files
            ]

        with upload_output:
            clear_output(wait=True)
            if uploaded_items:
                predict_uploaded_images(uploaded_items)
            else:
                print("Selecciona una o varias imágenes PNG o JPG.")

    upload_widget.observe(on_upload, names="value")
    display(
        widgets.VBox(
            [
                widgets.HTML(value="<b>Selecciona una o varias imágenes de dígitos:</b>"),
                upload_widget,
                upload_output,
            ]
        )
    )


## 8. Tabla final de la arquitectura

La tabla distingue la entrada de las capas `Dense`. Cuenta una capa de entrada y tres capas `Dense` entrenables. `ReLU` y `softmax` se aplican dentro de sus capas `Dense`; no se cuentan como capas adicionales. La columna de activación muestra cuál usa cada capa.


In [ ]:
# BLOQUE 1. Reunir las capas Dense y describir también la entrada del modelo.
from html import escape

dense_layers = [
    layer for layer in model.layers if isinstance(layer, tf.keras.layers.Dense)
]
input_units = int(np.prod(model.input_shape[1:]))
layer_rows = [
    {
        "Capa": "entrada",
        "Tipo": "Input",
        "Neuronas / unidades": f"{input_units} características",
        "Activación": "Ninguna",
        "Forma de salida": str(model.input_shape),
        "Parámetros": 0,
    }
]

for layer in dense_layers:
    layer_rows.append(
        {
            "Capa": layer.name,
            "Tipo": type(layer).__name__,
            "Neuronas / unidades": int(layer.units),
            "Activación": layer.activation.__name__,
            "Forma de salida": str(tuple(layer.output.shape)),
            "Parámetros": layer.count_params(),
        }
    )

# BLOQUE 2. Verificar que la tabla describe la arquitectura definida arriba.
assert len(dense_layers) == 3
assert [int(layer.units) for layer in dense_layers] == [64, 16, 10]
assert [layer.activation.__name__ for layer in dense_layers] == ["relu", "relu", "softmax"]
assert int(model.output_shape[-1]) == 10

# BLOQUE 3. Renderizar la tabla como HTML para que se vea directamente en el cuaderno.
columns = list(layer_rows[0].keys())
header_html = "".join(f"<th>{escape(column)}</th>" for column in columns)
body_html = "".join(
    "<tr>"
    + "".join(f"<td>{escape(str(row[column]))}</td>" for column in columns)
    + "</tr>"
    for row in layer_rows
)
table_html = (
    '<table style="border-collapse:collapse;width:100%">'
    '<caption style="font-weight:bold;padding:8px">Arquitectura del modelo MNIST</caption>'
    f"<thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody></table>"
)
display(HTML(table_html))

print(f"Capas totales (entrada + Dense): {len(layer_rows)}")
print(f"Capas Dense entrenables: {len(dense_layers)}")
print(f"Neuronas en las capas Dense: {sum(int(layer.units) for layer in dense_layers)}")
print(f"Parámetros entrenables: {model.count_params():,}")
